# Shapley Methods — Cross-Dataset Summary

Aggregated metrics across all 4 synthetic datasets. Reuses helper functions from `analysis_utils.py`.

| Dataset | Confounders | Data type |
|---------|------------|-----------|
| `linear_conf_f50_s1000_p30` | Yes | Linear |
| `linear_no_conf_f50_s1000_p30` | No | Linear |
| `mixed_conf_f50_s1000_30` | Yes | Mixed |
| `mixed_no_conf_f50_s1000_p30` | No | Mixed |

**Metrics computed:**
- **Spearman ρ** vs Traditional — feature ranking correlation
- **Top-K Jaccard** vs Traditional — feature set overlap at K = 5, 10, 20
- **True-parent Precision@K** — fraction of top-K that are true Y-parents
- **GSS** — Graph Sensitivity Score (PC vs LiNGAM magnitude shift)
- **Sign Alignment** vs True graph — fraction of instances with matching sign
- **Magnitude TGA** vs True graph — relative magnitude deviation

In [1]:
import sys
import importlib
from pathlib import Path as _Path
sys.path.insert(0, str(_Path("..").resolve()))

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import json
from pathlib import Path
import warnings
warnings.filterwarnings("ignore")

from scipy.stats import spearmanr

# Force reload so any changes to analysis_utils.py are picked up
if "analysis_utils" in sys.modules:
    importlib.reload(sys.modules["analysis_utils"])

from analysis_utils import (
    mean_abs_shap, top_k_jaccard,
    compute_sign_alignment, compute_tga, compute_sss,
    compute_gss, top_k_features, rank_shift_top_k,
    flow_graph_stats,
    build_dag_graph, make_dag_pos, draw_dag_highlight,
)

# ── Paths ─────────────────────────────────────────────────────────────────────
BASE_DIR           = Path("..").resolve()
EXPLAINABILITY_DIR = BASE_DIR / "data" / "explainability"
CAUSAL_DIR         = BASE_DIR / "data" / "causal"
PROCESSED_DIR      = BASE_DIR / "data" / "processed"
SYNTHETIC_DIR      = BASE_DIR / "data" / "synthetic"

# ── Datasets to compare ───────────────────────────────────────────────────────
DATASETS = [
    "linear_conf_f50_s1000_p30",
    # "linear_no_conf_f50_s1000_p30",
    # "mixed_conf_f50_s1000_p30",
    # "mixed_no_conf_f50_s1000_p30",
]
MODEL = "lgbm"
K_VALUES     = [5, 10, 20]
BASE_METHODS = ["Asymmetric", "Causal", "Flow"]
DISC_GRAPHS  = ["PC", "LiNGAM"]

METHOD_COLORS = {
    "Scratch":    "#636363",
    "Asymmetric": "#2166ac",
    "Causal":     "#4dac26",
    "Flow":       "#d01c8b",
}
# Short dataset labels for display
DS_LABELS = {
    "linear_conf_f50_s1000_p30":     "Lin-Conf",
    "linear_no_conf_f50_s1000_p30":  "Lin-NoConf",
    "mixed_conf_f50_s1000_p30":      "Mix-Conf",
    "mixed_no_conf_f50_s1000_p30":   "Mix-NoConf",
}

print("Setup complete. Datasets:", DATASETS)


Setup complete. Datasets: ['linear_conf_f50_s1000_p30']


## 1. Load All Data

Load Shapley values, causal graphs, metadata, and pre-computed true-graph arrays for every dataset.

In [2]:
def load_dataset(dataset: str, model: str = "lgbm") -> dict:
    """
    Load all Shapley values, causal graphs, metadata, and true-graph arrays
    for one dataset. Returns a dict with all data needed for metric computation.
    """
    exp_base = EXPLAINABILITY_DIR / dataset / model
    causal   = CAUSAL_DIR

    # ── Main SHAP data (from pipeline output) ─────────────────────────────
    keys = {
        "Scratch":             exp_base / "scratch"  / "shapley_values.npy",
        "Asymmetric (PC)":     exp_base / "pc"        / "asymmetric" / "shapley_values.npy",
        "Causal (PC)":         exp_base / "pc"        / "causal"     / "shapley_values.npy",
        "Flow (PC)":           exp_base / "pc"        / "flow"       / "shapley_values.npy",
        "Asymmetric (LiNGAM)": exp_base / "lingam"   / "asymmetric" / "shapley_values.npy",
        "Causal (LiNGAM)":     exp_base / "lingam"   / "causal"     / "shapley_values.npy",
        "Flow (LiNGAM)":       exp_base / "lingam"   / "flow"       / "shapley_values.npy",
    }
    subset_keys = {
        "Asymmetric (PC)":     exp_base / "pc"        / "asymmetric" / "shapley_values.npy",
        "Causal (PC)":         exp_base / "pc"        / "causal"     / "shapley_values.npy",
        "Flow (PC)":           exp_base / "pc"        / "flow"       / "shapley_values.npy",
        "Asymmetric (LiNGAM)": exp_base / "lingam"   / "asymmetric" / "shapley_values.npy",
        "Causal (LiNGAM)":     exp_base / "lingam"   / "causal"     / "shapley_values.npy",
        "Flow (LiNGAM)":       exp_base / "lingam"   / "flow"       / "shapley_values.npy",
        "Asymmetric (True)":   exp_base / "true"      / "asymmetric" / "shapley_values.npy",
        "Causal (True)":       exp_base / "true"      / "causal"     / "shapley_values.npy",
        "Flow (True)":         exp_base / "true"      / "flow"       / "shapley_values.npy",
    }

    shap_data        = {k: np.load(v) for k, v in keys.items()        if v.exists()}
    subset_shap_data = {k: np.load(v) for k, v in subset_keys.items() if v.exists()}

    # ── Feature names ──────────────────────────────────────────────────────
    with open(causal / f"{dataset}_pc_results.json") as f:
        pc_results = json.load(f)
    with open(causal / f"{dataset}_lingam_results.json") as f:
        lingam_results = json.load(f)
    feature_names = [n for n in pc_results["feature_names"] if n != "Y"]

    # ── True Y-parents ─────────────────────────────────────────────────────
    with open(SYNTHETIC_DIR / f"{dataset}_metadata.json") as f:
        meta = json.load(f)
    y_parent_set = set(meta["y_parent_indices"])

    # ── Graph stats ────────────────────────────────────────────────────────
    pc_edges,     pc_sources     = flow_graph_stats(pc_results["adjacency_matrix"],     len(feature_names))
    lingam_edges, lingam_sources = flow_graph_stats(lingam_results["adjacency_matrix"], len(feature_names))

    return dict(
        shap_data        = shap_data,
        subset_shap_data = subset_shap_data,
        feature_names    = feature_names,
        y_parent_set     = y_parent_set,
        pc_edges         = pc_edges,
        lingam_edges     = lingam_edges,
    )


# Load all datasets
all_data = {}
for ds in DATASETS:
    all_data[ds] = load_dataset(ds)
    n_shap = all_data[ds]["shap_data"]["Scratch"].shape[0]
    n_feat = len(all_data[ds]["feature_names"])
    n_par  = len(all_data[ds]["y_parent_set"])
    loaded = len(all_data[ds]["shap_data"])
    sub_loaded = len(all_data[ds]["subset_shap_data"])
    print(f"  {DS_LABELS[ds]:12s}  {loaded} main SHAP + {sub_loaded} subset  "
          f"n_shap={n_shap}  n_feat={n_feat}  true_parents={n_par}")

  Lin-Conf      7 main SHAP + 9 subset  n_shap=100  n_feat=50  true_parents=15


## 2. Compute All Metrics

For each dataset × method × graph, compute Spearman ρ, Top-K Jaccard, Precision@K, and GSS.

In [3]:
all_metrics = []   # list of dicts, one per (dataset, method, graph)
gss_metrics = []   # one per (dataset, method)
sss_metrics = []   # one per (dataset, method)
tga_metrics = []   # one per (dataset, method, graph)
sa_metrics  = []   # sign alignment, one per (dataset, method, graph)
sa_scratch_metrics  = []   # sign alignment vs Scratch
tga_scratch_metrics = []   # TGA vs Scratch

# Store per-feature arrays by dataset for diagnostics and visualization
gss_per_feat = {}   # dict[dataset] -> dict[method] -> ndarray(n_features,)
sss_per_feat = {}   # dict[dataset] -> dict[method] -> ndarray(n_features,)

for ds in DATASETS:
    d             = all_data[ds]
    shap_data     = d["shap_data"]
    subset_shap   = d["subset_shap_data"]
    feature_names = d["feature_names"]
    y_parent_set  = d["y_parent_set"]
    n_feat        = len(feature_names)
    random_base   = len(y_parent_set) / n_feat

    scratch_ma = mean_abs_shap(shap_data, "Scratch")

    # ── Spearman ρ, Jaccard, Precision@K ──────────────────────────────────
    for meth in BASE_METHODS:
        for g in DISC_GRAPHS:
            key = f"{meth} ({g})"
            if key not in shap_data:
                continue
            ma = mean_abs_shap(shap_data, key)
            rho, _ = spearmanr(ma, scratch_ma)
            for k in K_VALUES:
                jacc = top_k_jaccard(ma, scratch_ma, k)
                top_k_idx = set(np.argsort(ma)[-k:])
                prec = len(top_k_idx & y_parent_set) / k
                all_metrics.append(dict(
                    Dataset     = DS_LABELS[ds],
                    DatasetFull = ds,
                    Method      = meth,
                    Graph       = g,
                    K           = k,
                    Spearman    = float(rho),
                    Jaccard     = float(jacc),
                    Precision   = float(prec),
                    RandomBase  = float(random_base),
                ))

    # ── Scratch Precision@K (reference) ───────────────────────────────────
    scratch_ma_arr = mean_abs_shap(shap_data, "Scratch")
    for k in K_VALUES:
        top_k_idx = set(np.argsort(scratch_ma_arr)[-k:])
        prec = len(top_k_idx & y_parent_set) / k
        all_metrics.append(dict(
            Dataset="Scratch", DatasetFull=ds, Method="Scratch", Graph="—",
            K=k, Spearman=np.nan, Jaccard=np.nan, Precision=float(prec),
            RandomBase=float(random_base),
        ))

    # ── GSS — using standard compute_gss function ─────────────────────────
    # Returns per-feature arrays: dict[method] -> ndarray(n_features,)
    gss_feat = compute_gss(shap_data, BASE_METHODS)
    gss_per_feat[ds] = gss_feat  # Store for diagnostics and visualization
    for meth, feat_arr in gss_feat.items():
        gss_metrics.append(dict(
            Dataset   = DS_LABELS[ds],
            Method    = meth,
            MeanGSS   = float(np.mean(feat_arr)),
            MedianGSS = float(np.median(feat_arr)),
        ))

    # ── SSS — using standard compute_sss function ─────────────────────────
    # Returns per-feature arrays: dict[method] -> ndarray(n_features,)
    sss_feat = compute_sss(shap_data, BASE_METHODS)
    sss_per_feat[ds] = sss_feat  # Store for diagnostics and visualization
    for meth, feat_arr in sss_feat.items():
        sss_metrics.append(dict(
            Dataset = DS_LABELS[ds],
            Method  = meth,
            MeanSSS = float(np.nanmean(feat_arr)),
        ))

    # ── Sign Alignment vs True ─────────────────────────────────────────────
    # Returns per-feature arrays: dict[(method, graph)] -> ndarray(n_features,)
    sign_align = compute_sign_alignment(subset_shap, BASE_METHODS, DISC_GRAPHS, reference="True")
    for (meth, g), arr in sign_align.items():
        sa_metrics.append(dict(
            Dataset       = DS_LABELS[ds],
            Method        = meth,
            Graph         = g,
            MeanSignAlign = float(np.nanmean(arr)),
        ))

    # ── Magnitude TGA vs True ──────────────────────────────────────────────
    # Returns per-feature arrays: dict[(method, graph)] -> ndarray(n_features,)
    tga_data, _ = compute_tga(subset_shap, feature_names, BASE_METHODS, DISC_GRAPHS, reference="True")
    for (meth, g), arr in tga_data.items():
        tga_metrics.append(dict(
            Dataset = DS_LABELS[ds],
            Method  = meth,
            Graph   = g,
            MeanTGA = float(np.mean(arr)),
        ))

    # ── Sign Alignment vs Scratch (Traditional baseline) ──────────────────
    # Merge subset SHAP + Scratch as reference
    combined = dict(subset_shap)
    if "Scratch" in shap_data:
        combined["Scratch"] = shap_data["Scratch"]

    sign_align_scratch = compute_sign_alignment(
        combined, BASE_METHODS, DISC_GRAPHS, reference="Scratch"
    )
    for (meth, g), arr in sign_align_scratch.items():
        sa_scratch_metrics.append(dict(
            Dataset       = DS_LABELS[ds],
            Method        = meth,
            Graph         = g,
            MeanSignAlign = float(np.nanmean(arr)),
        ))

    # ── Magnitude TGA vs Scratch (Traditional baseline) ───────────────────
    tga_scratch, _ = compute_tga(
        combined, feature_names, BASE_METHODS, DISC_GRAPHS, reference="Scratch"
    )
    for (meth, g), arr in tga_scratch.items():
        tga_scratch_metrics.append(dict(
            Dataset = DS_LABELS[ds],
            Method  = meth,
            Graph   = g,
            MeanTGA = float(np.nanmean(arr)),
        ))

# ── Build summary DataFrames ───────────────────────────────────────────────
df_metrics     = pd.DataFrame(all_metrics)
df_gss         = pd.DataFrame(gss_metrics)
df_sss         = pd.DataFrame(sss_metrics)
df_sa          = pd.DataFrame(sa_metrics)
df_tga         = pd.DataFrame(tga_metrics)
df_sa_scratch  = pd.DataFrame(sa_scratch_metrics)
df_tga_scratch = pd.DataFrame(tga_scratch_metrics)

print(f"Metrics computed:")
print(f"  All metrics: {len(df_metrics)} rows")
print(f"  GSS: {len(df_gss)} rows (per-feature arrays stored in gss_per_feat)")
print(f"  SSS: {len(df_sss)} rows (per-feature arrays stored in sss_per_feat)")
print(f"  Sign Alignment vs True: {len(df_sa)} rows")
print(f"  TGA vs True: {len(df_tga)} rows")
print(f"  Sign Alignment vs Scratch: {len(df_sa_scratch)} rows")
print(f"  TGA vs Scratch: {len(df_tga_scratch)} rows")

Metrics computed:
  All metrics: 21 rows
  GSS: 3 rows (per-feature arrays stored in gss_per_feat)
  SSS: 3 rows (per-feature arrays stored in sss_per_feat)
  Sign Alignment vs True: 6 rows
  TGA vs True: 6 rows
  Sign Alignment vs Scratch: 6 rows
  TGA vs Scratch: 6 rows


In [4]:
def feature_profile(feature_name: str, method_key: str, dataset: str = None):
    """
    Print a full metric profile for one feature under one method/graph combination
    and return a rank-comparison DataFrame across all loaded methods.

    Parameters
    ----------
    feature_name : str   Feature name, e.g. "X_5"
    method_key   : str   Method key, e.g. "Asymmetric (PC)", "Causal (LiNGAM)", "Scratch"
    dataset      : str   Dataset name from DATASETS, or None to loop over all datasets.

    Returns
    -------
    rank_df : pd.DataFrame
        Rank of this feature across all loaded method keys (last dataset if multiple).
    """
    # ── Parse base method and graph from key ─────────────────────────────────
    if " (" in method_key and method_key.endswith(")"):
        base_meth = method_key[: method_key.rfind(" (")]
        graph     = method_key[method_key.rfind("(") + 1 : -1]
    else:
        base_meth = method_key
        graph     = None

    ds_list  = [dataset] if dataset else DATASETS
    rank_df  = pd.DataFrame()
    W        = 72

    for ds in ds_list:
        d             = all_data[ds]
        shap_d        = d["shap_data"]
        subset_shap_d = d["subset_shap_data"]
        feat_names    = d["feature_names"]
        y_parent_set  = d["y_parent_set"]
        n_feat        = len(feat_names)

        if feature_name not in feat_names:
            print(f"[{DS_LABELS[ds]}] Feature '{feature_name}' not found. "
                  f"Available: {feat_names[:5]} ...")
            continue

        feat_idx = feat_names.index(feature_name)
        arr_full = shap_d.get(method_key)
        if arr_full is None:
            arr_full = subset_shap_d.get(method_key)

        if arr_full is None:
            all_keys = list(shap_d) + [k for k in subset_shap_d if k not in shap_d]
            print(f"[{DS_LABELS[ds]}] Method key '{method_key}' not found. "
                  f"Available: {sorted(all_keys)}")
            continue

        # ── Core attribution ──────────────────────────────────────────────────
        feat_shap   = arr_full[:, feat_idx]
        mean_signed = float(feat_shap.mean())
        mean_abs    = float(np.abs(feat_shap).mean())
        is_y_parent = feat_idx in y_parent_set

        ma_all    = np.abs(arr_full).mean(axis=0)
        rank_this = int(np.argsort(np.argsort(-ma_all))[feat_idx]) + 1

        # Rank in Scratch
        rank_scratch = None
        if "Scratch" in shap_d:
            scratch_ma   = np.abs(shap_d["Scratch"]).mean(axis=0)
            rank_scratch = int(np.argsort(np.argsort(-scratch_ma))[feat_idx]) + 1

        delta_rank = (rank_this - rank_scratch) if rank_scratch is not None else None

        # ── Graph Sensitivity (GSS + SSS) ─────────────────────────────────────
        gss_val = sss_val = None
        if base_meth in BASE_METHODS:
            gss_map = compute_gss(shap_d, [base_meth])
            if base_meth in gss_map:
                gss_val = float(gss_map[base_meth][feat_idx])
            sss_map = compute_sss(shap_d, [base_meth])
            if base_meth in sss_map:
                sss_val = float(sss_map[base_meth][feat_idx])

        # ── Alignment vs True DAG ─────────────────────────────────────────────
        sa_true_val = tga_true_val = None
        if graph and graph not in ("Scratch",) and base_meth in BASE_METHODS:
            sa_true_map = compute_sign_alignment(
                subset_shap_d, [base_meth], [graph], reference="True")
            tga_true_map, _ = compute_tga(
                subset_shap_d, feat_names, [base_meth], [graph], reference="True")
            if (base_meth, graph) in sa_true_map:
                sa_true_val  = float(sa_true_map[(base_meth, graph)][feat_idx])
            if (base_meth, graph) in tga_true_map:
                tga_true_val = float(tga_true_map[(base_meth, graph)][feat_idx])

        # ── Alignment vs Scratch ───────────────────────────────────────────────
        sa_scratch_val = tga_scratch_val = None
        if graph and base_meth in BASE_METHODS:
            combined_d = {**subset_shap_d}
            if "Scratch" in shap_d:
                combined_d["Scratch"] = shap_d["Scratch"]
            sa_scr_map = compute_sign_alignment(
                combined_d, [base_meth], [graph], reference="Scratch")
            tga_scr_map, _ = compute_tga(
                combined_d, feat_names, [base_meth], [graph], reference="Scratch")
            if (base_meth, graph) in sa_scr_map:
                sa_scratch_val  = float(sa_scr_map[(base_meth, graph)][feat_idx])
            if (base_meth, graph) in tga_scr_map:
                tga_scratch_val = float(tga_scr_map[(base_meth, graph)][feat_idx])

        # ── Rank comparison across all loaded method keys ─────────────────────
        all_keys = list(shap_d.keys()) + [k for k in subset_shap_d if k not in shap_d]
        rank_rows = []
        for k in sorted(all_keys):
            src    = shap_d if k in shap_d else subset_shap_d
            arr_k  = src[k]
            ma_k   = np.abs(arr_k).mean(axis=0)
            ms_k   = arr_k[:, feat_idx].mean()
            rk     = int(np.argsort(np.argsort(-ma_k))[feat_idx]) + 1
            rank_rows.append({
                "Method":       k,
                "Rank":         rk,
                "Mean SHAP":    round(float(ms_k), 6),
                "Mean |SHAP|":  round(float(ma_k[feat_idx]), 6),
                "Current →":    "←" if k == method_key else "",
            })
        rank_df = (pd.DataFrame(rank_rows)
                     .sort_values("Mean |SHAP|", ascending=False)
                     .reset_index(drop=True))
        rank_df.index += 1

        # ── Print ─────────────────────────────────────────────────────────────
        print(f"\n{'═' * W}")
        print(f"  FEATURE PROFILE: {feature_name}  |  {method_key}  |  {DS_LABELS[ds]}")
        print(f"{'═' * W}")

        print(f"\n  CORE ATTRIBUTION")
        print(f"    Mean SHAP (signed)  :  {mean_signed:+.6f}")
        print(f"    Mean |SHAP|         :   {mean_abs:.6f}  →  Rank {rank_this}/{n_feat}")
        print(f"    Is Y-parent (true)  :   {'Yes ✓' if is_y_parent else 'No'}")
        if rank_scratch is not None:
            sign = "+" if delta_rank > 0 else ""
            print(f"    Rank vs Scratch     :   {rank_scratch}  →  {rank_this}"
                  f"  ({sign}{delta_rank}  {'↓ less important' if delta_rank > 0 else '↑ more important' if delta_rank < 0 else '= no change'})")

        if gss_val is not None or sss_val is not None:
            print(f"\n  GRAPH SENSITIVITY  ({base_meth}: PC vs LiNGAM)")
            if gss_val is not None:
                dir_str = "PC assigns more weight" if gss_val > 0 else "LiNGAM assigns more weight"
                print(f"    GSS (magnitude)    :  {gss_val:+.6f}  ({dir_str})")
            if sss_val is not None:
                stab = "stable" if sss_val >= 0.8 else ("unstable" if sss_val < 0.5 else "moderate")
                print(f"    SSS (sign)         :   {sss_val:.4f}  ({stab})")

        if sa_true_val is not None or tga_true_val is not None:
            print(f"\n  ALIGNMENT VS TRUE DAG  ({base_meth} — {graph})")
            if sa_true_val is not None:
                agree = "agrees" if sa_true_val >= 0.8 else ("disagrees" if sa_true_val < 0.5 else "partially agrees")
                print(f"    Sign Alignment     :   {sa_true_val:.4f}  ({agree} with true oracle)")
            if tga_true_val is not None:
                oe = "over-estimates" if tga_true_val > 0 else "under-estimates"
                print(f"    TGA                :  {tga_true_val:+.6f}  ({oe} true-graph magnitude)")

        if sa_scratch_val is not None or tga_scratch_val is not None:
            print(f"\n  ALIGNMENT VS SCRATCH  ({base_meth} — {graph})")
            if sa_scratch_val is not None:
                agree = "agrees" if sa_scratch_val >= 0.8 else ("disagrees" if sa_scratch_val < 0.5 else "partially agrees")
                print(f"    Sign Alignment     :   {sa_scratch_val:.4f}  ({agree} with scratch baseline)")
            if tga_scratch_val is not None:
                oe = "over-estimates" if tga_scratch_val > 0 else "under-estimates"
                print(f"    TGA                :  {tga_scratch_val:+.6f}  ({oe} scratch magnitude)")

        print(f"\n  RANK COMPARISON  (all methods, feature: {feature_name})")
        display(rank_df)

    return rank_df


# ── Example usage ─────────────────────────────────────────────────────────────
# Replace with any feature name and method key visible in shap_data / subset_shap_data:
#
#   feature_profile("X_3",  "Asymmetric (PC)")
#   feature_profile("X_17", "Causal (LiNGAM)",  dataset="linear_conf_f50_s1000_p50")
#   feature_profile("X_42", "Scratch")

## 3. Cross-Dataset Comparison

### 3a — Spearman ρ vs Traditional  
Feature ranking correlation between each causal method and the graph-free Traditional baseline.

### 3b — Top-K Jaccard vs Traditional  
Fraction of the top-K features shared with Traditional's top-K set.

### 3c — True-Parent Precision@K  
Fraction of the top-K features that are true causal parents of Y.


In [5]:
# ── 3a: Spearman ρ heatmap — method×graph rows, dataset columns ──────────────
df_rho = df_metrics[df_metrics["Method"].isin(BASE_METHODS)].drop_duplicates(
    subset=["Dataset", "DatasetFull", "Method", "Graph", "Spearman"]
)[["DatasetFull", "Method", "Graph", "Spearman"]].dropna()
# Take one row per (dataset, method, graph) — Spearman doesn't depend on K
df_rho = df_rho.groupby(["DatasetFull", "Method", "Graph"], as_index=False)["Spearman"].first()

pivot_rho = df_rho.pivot_table(
    index=["Method", "Graph"], columns="DatasetFull", values="Spearman"
)
pivot_rho = pivot_rho[[ds for ds in DATASETS if ds in pivot_rho.columns]]

row_labels = [f"{m} ({g})" for m, g in pivot_rho.index]
col_labels = [DS_LABELS[c] for c in pivot_rho.columns]

fig_rho = go.Figure(go.Heatmap(
    z            = pivot_rho.values,
    x            = col_labels,
    y            = row_labels,
    colorscale   = "RdYlGn",
    zmin=0.5, zmax=1.0,
    text         = [[f"{v:.3f}" for v in row] for row in pivot_rho.values],
    texttemplate = "%{text}",
    textfont     = dict(size=12),
    colorbar     = dict(title=dict(text="Spearman ρ", font=dict(size=11))),
    hovertemplate="<b>%{y}</b> — %{x}<br>ρ = %{z:.3f}<extra></extra>",
))
fig_rho.update_layout(
    title=dict(text="Spearman Structure Aware ρ vs Traditional — All Datasets<br>"
               "<sup>1.0 = same feature ranking as graph-free baseline</sup>",
               font=dict(size=13)),
    height=320, width=700,
    margin=dict(l=160, r=80, t=80, b=60),
    xaxis=dict(tickfont=dict(size=11), tickangle=-20),
    yaxis=dict(tickfont=dict(size=10)),
)
fig_rho.show()


In [6]:
# ── 3c: True-parent Precision@K — one subplot per K-value ───────────────────
fig_prec = make_subplots(
    rows=1, cols=len(K_VALUES),
    subplot_titles=[f"Precision@{k}" for k in K_VALUES],
    shared_yaxes=True,
)
LEGEND_SEEN2 = set()
combos = [(m, g) for m in BASE_METHODS for g in DISC_GRAPHS]

for ci, k in enumerate(K_VALUES, start=1):
    sub = df_metrics[(df_metrics["K"] == k) & (df_metrics["Method"].isin(BASE_METHODS))]
    for meth, g in combos:
        rows = sub[(sub["Method"] == meth) & (sub["Graph"] == g)]
        if rows.empty:
            continue
        name = f"{meth} ({g})"
        show = name not in LEGEND_SEEN2
        LEGEND_SEEN2.add(name)
        fig_prec.add_trace(
            go.Bar(
                x           = [DS_LABELS[rows.iloc[i]["DatasetFull"]] for i in range(len(rows))],
                y           = rows["Precision"].tolist(),
                name        = name,
                legendgroup = name,
                showlegend  = show,
                marker_color= METHOD_COLORS[meth],
                opacity     = 0.7 if g == "LiNGAM" else 1.0,
                hovertemplate=f"<b>{name}</b><br>Prec=%{{y:.3f}}<extra></extra>",
            ),
            row=1, col=ci,
        )
    # Random baseline line
    random_vals = df_metrics[(df_metrics["K"] == k) & (df_metrics["Method"].isin(BASE_METHODS))]
    if not random_vals.empty:
        xs = [DS_LABELS[ds] for ds in DATASETS]
        fig_prec.add_trace(
            go.Scatter(
                x=xs, y=[random_vals.groupby("DatasetFull")["RandomBase"].first().get(ds, 0) for ds in DATASETS],
                mode="lines", line=dict(dash="dash", color="black", width=1.2),
                name="Random", legendgroup="Random", showlegend=(ci == 1),
                hovertemplate="Random baseline<br>Prec=%{y:.3f}<extra></extra>",
            ),
            row=1, col=ci,
        )

fig_prec.update_layout(
    title=dict(text="True-Parent Precision@K<br><sup>Dashed black line = random baseline (n_parents/n_features)</sup>",
               font=dict(size=13)),
    height=360, width=900,
    barmode="group",
    legend=dict(x=1.01, y=1, xanchor="left", font=dict(size=10)),
    margin=dict(l=60, r=200, t=80, b=80),
)
fig_prec.update_yaxes(range=[0, 1.05], title_text="Precision", col=1)
fig_prec.update_xaxes(tickangle=-25)
fig_prec.show()

## 4. Graph Sensitivity Score (GSS)

How much does each method's output shift between the PC and LiNGAM graphs?  
Lower is more stable under graph uncertainty. GSS = 1 means the largest shift across all methods.

In [7]:
# GSS grouped bar — dataset × method
fig_gss = go.Figure()
for meth in BASE_METHODS:
    sub = df_gss[df_gss["Method"] == meth]
    fig_gss.add_trace(go.Bar(
        name         = meth,
        x            = sub["Dataset"].tolist(),
        y            = sub["MeanGSS"].tolist(),
        marker_color = METHOD_COLORS[meth],
        error_y      = None,
        hovertemplate=f"<b>{meth}</b><br>Dataset=%{{x}}<br>MeanGSS=%{{y:.4f}}<extra></extra>",
    ))

fig_gss.update_layout(
    title=dict(text="Graph Sensitivity Score (Mean GSS) — Near 0 = PC and LiNGAM Agree<br>"
               "<sup>Signed magnitude difference: positive = PC higher, negative = LiNGAM higher</sup>",
               font=dict(size=13)),
    barmode="group",
    yaxis=dict(title="Mean GSS", zeroline=True, zerolinewidth=1.5, zerolinecolor="gray"),
    height=380, width=700,
    legend=dict(x=0.01, y=0.99, xanchor="left", font=dict(size=11)),
    margin=dict(l=70, r=30, t=90, b=80),
    xaxis=dict(tickangle=-20),
)
fig_gss.show()


## 4b. Sign Stability Score (SSS) & Cross-Evaluation with GSS

**SSS** is the sign-domain analogue of the True-Graph Alignment sign metric: for each (instance, feature) pair we ask whether the *sign* of the SHAP value **agrees** between the PC and LiNGAM graph — consistent with how `compute_sign_alignment` compares a discovered graph against the True DAG.

$$\text{SSS}(m, f) = \frac{1}{|\{i : \phi^{PC}_{i,f} \neq 0 \;\wedge\; \phi^{LiNGAM}_{i,f} \neq 0\}|} \sum_i \mathbf{1}\!\left[\text{sign}\!\left(\phi^{PC}_{i,f}\right) = \text{sign}\!\left(\phi^{LiNGAM}_{i,f}\right)\right]$$

Higher SSS = signs are **more consistent** between the two discovered graphs (more stable).

The **cross-evaluation scatter** places every (method × dataset) in a common stability space where both axes measure instability (lower = more stable):
- **x-axis — GSS**: magnitude instability between PC and LiNGAM  
- **y-axis — (1 − SSS)**: sign flip rate between PC and LiNGAM  
- **Bottom-left corner** = maximally stable across both dimensions.

In [8]:
# ── Compute SSS via analysis_utils.compute_sss ────────────────────────────────
# Returns dict: method → ndarray(n_features,) agreement rate in [0, 1].
# Higher = signs more consistent between PC and LiNGAM graphs.

sss_metrics = []

for ds in DATASETS:
    d         = all_data[ds]
    shap_data = d["shap_data"]

    sss_per_method = compute_sss(shap_data, BASE_METHODS)

    for meth, feat_arr in sss_per_method.items():
        sss_metrics.append(dict(
            Dataset = DS_LABELS[ds],
            Method  = meth,
            MeanSSS = float(np.nanmean(feat_arr)),
        ))

df_sss = pd.DataFrame(sss_metrics)
print(df_sss.to_string(index=False))

 Dataset     Method  MeanSSS
Lin-Conf Asymmetric 0.940709
Lin-Conf     Causal 0.620600
Lin-Conf       Flow 0.867267


In [9]:
# ── SSS bar chart ─────────────────────────────────────────────────────────────
fig_sss = go.Figure()
for meth in BASE_METHODS:
    sub = df_sss[df_sss["Method"] == meth]
    fig_sss.add_trace(go.Bar(
        name         = meth,
        x            = sub["Dataset"].tolist(),
        y            = sub["MeanSSS"].tolist(),
        marker_color = METHOD_COLORS[meth],
        hovertemplate=f"<b>{meth}</b><br>Dataset=%{{x}}<br>MeanSSS=%{{y:.3f}}<extra></extra>",
    ))

fig_sss.update_layout(
    title=dict(text="Sign Stability Score (Mean SSS) — Higher = More Stable<br>"
               "<sup>Fraction of (instance, feature) pairs where SHAP sign agrees between PC and LiNGAM</sup>",
               font=dict(size=13)),
    barmode="group",
    yaxis=dict(title="Mean SSS (sign agreement)", range=[0, 1.05]),
    height=380, width=700,
    legend=dict(x=0.01, y=0.01, xanchor="left", font=dict(size=11)),
    margin=dict(l=70, r=30, t=90, b=80),
    xaxis=dict(tickangle=-20),
)
fig_sss.show()

In [10]:
# ── Cross-evaluation: GSS vs (1 − SSS) scatter ────────────────────────────────
# x: GSS (signed)   — positive = PC assigns higher importance on average
#                     negative = LiNGAM assigns higher importance on average
# y: 1 − MeanSSS    — sign flip rate between PC and LiNGAM
df_cross = df_gss[["Dataset", "Method", "MeanGSS"]].merge(
    df_sss[["Dataset", "Method", "MeanSSS"]],
    on=["Dataset", "Method"],
)
df_cross["SignFlipRate"] = 1 - df_cross["MeanSSS"]

fig_cross = go.Figure()
SEEN_METHODS = set()

for _, row in df_cross.iterrows():
    meth = row["Method"]
    ds   = row["Dataset"]
    show_meth = meth not in SEEN_METHODS
    SEEN_METHODS.add(meth)

    fig_cross.add_trace(go.Scatter(
        x    = [row["MeanGSS"]],
        y    = [row["SignFlipRate"]],
        mode = "markers",
        name = meth,
        legendgroup = meth,
        showlegend  = show_meth,
        marker = dict(
            color  = METHOD_COLORS[meth],
            symbol = "circle",
            size   = 13,
            line   = dict(width=1.2, color="white"),
        ),
        hovertemplate=(
            f"<b>{meth}</b><br>"
            f"Dataset: {ds}<br>"
            "GSS (signed magnitude bias): %{x:.4f}<br>"
            "Sign flip rate (1−SSS): %{y:.3f}<extra></extra>"
        ),
    ))

# Symmetric x-axis range around 0 (GSS is signed)
gss_abs_max = df_cross["MeanGSS"].abs().max() * 1.3
y_max       = df_cross["SignFlipRate"].max() * 1.2

# Vertical reference line at x=0 (zero bias between PC and LiNGAM)
fig_cross.add_vline(
    x=0, line=dict(dash="dot", color="lightgray", width=1.5)
)
fig_cross.add_annotation(
    x=0, y=y_max * 0.97, text="PC = LiNGAM", showarrow=False,
    font=dict(size=9, color="#aaaaaa"), xanchor="center",
)

# Quadrant labels
for qx, qy, label in [
    (-gss_abs_max * 0.6, y_max * 0.75, "LiNGAM > PC<br>Sign-unstable"),
    ( gss_abs_max * 0.6, y_max * 0.75, "PC > LiNGAM<br>Sign-unstable"),
    (-gss_abs_max * 0.6, y_max * 0.15, "LiNGAM > PC<br>Sign-stable"),
    ( gss_abs_max * 0.6, y_max * 0.15, "PC > LiNGAM<br>Sign-stable"),
]:
    fig_cross.add_annotation(
        x=qx, y=qy, text=label, showarrow=False,
        font=dict(size=9, color="#bbbbbb"), align="center",
    )

fig_cross.update_layout(
    title=dict(
        text="GSS (Signed) vs Sign Flip Rate (1 − SSS) — Magnitude Bias vs Sign Instability (PC ↔ LiNGAM)<br>"
             "<sup>x>0: PC assigns higher importance on avg; x<0: LiNGAM higher | Colour = method</sup>",
        font=dict(size=12),
    ),
    xaxis=dict(
        title="GSS — Signed Magnitude Bias",
        range=[-gss_abs_max, gss_abs_max],
        zeroline=True, zerolinewidth=1.5, zerolinecolor="gray",
    ),
    yaxis=dict(
        title="1 − SSS — Sign Flip Rate",
        range=[0, y_max],
        zeroline=False,
    ),
    height=480, width=660,
    legend=dict(x=1.02, y=1, xanchor="left", font=dict(size=11)),
    margin=dict(l=70, r=200, t=90, b=70),
)
fig_cross.show()


In [11]:
# ── Summary Table: Mean Metrics by Method (GSS vs 1-SSS) ────────────────────
df_cross_agg = df_cross.groupby("Method", as_index=False).agg({
    "MeanGSS": "mean",
    "MeanSSS": "mean",
    "SignFlipRate": "mean"
})
df_cross_agg = df_cross_agg.rename(columns={
    "MeanGSS": "Mean GSS",
    "MeanSSS": "Mean SSS",
    "SignFlipRate": "Mean Sign Flip Rate (1−SSS)"
})
df_cross_agg = df_cross_agg.sort_values("Method")

print("\nSummary: Mean Metrics PC vs LiNGAM Stability (averaged across all datasets)")
print("=" * 80)
display(df_cross_agg.round(4))


Summary: Mean Metrics PC vs LiNGAM Stability (averaged across all datasets)


,Method,Mean GSS,Mean SSS,Mean Sign Flip Rate (1−SSS)
0,Asymmetric,-0.0005,0.9407,0.0593
1,Causal,0.0070,0.6206,0.3794
2,Flow,0.0112,0.8673,0.1327


## 5. True-Graph Alignment

### 5a — Sign Alignment vs True DAG  
Mean fraction of instances where the causal method's SHAP sign matches the True-graph output per feature.  
Higher = more aligned with the oracle graph.

### 5b — Magnitude TGA vs True DAG  
Mean relative magnitude deviation vs True-graph output per feature.  
Lower = closer to oracle magnitude.

In [12]:
# ── 5a: Sign Alignment — faceted heatmap dataset × method per graph ──────────
for g in DISC_GRAPHS:
    sub = df_sa[df_sa["Graph"] == g]
    pivot = sub.pivot_table(index="Method", columns="Dataset", values="MeanSignAlign")
    pivot = pivot[[DS_LABELS[ds] for ds in DATASETS if DS_LABELS[ds] in pivot.columns]]

    row_labels = pivot.index.tolist()
    col_labels = pivot.columns.tolist()

    fig_sa = go.Figure(go.Heatmap(
        z            = pivot.values,
        x            = col_labels,
        y            = row_labels,
        colorscale   = "Blues",
        zmin=0.5, zmax=1.0,
        text         = [[f"{v:.3f}" if not np.isnan(v) else "—" for v in row] for row in pivot.values],
        texttemplate = "%{text}",
        textfont     = dict(size=12),
        colorbar     = dict(title=dict(text="Sign Align", font=dict(size=11))),
        hovertemplate=f"<b>%{{y}}</b> ({g}) — %{{x}}<br>Sign Align=%{{z:.3f}}<extra></extra>",
    ))
    fig_sa.update_layout(
        title=dict(text=f"Sign Alignment vs True DAG — {g} graph<br>"
                   "<sup>Fraction of instances where sign matches oracle</sup>",
                   font=dict(size=12)),
        height=260, width=700,
        margin=dict(l=130, r=80, t=70, b=60),
        xaxis=dict(tickfont=dict(size=11), tickangle=-15),
        yaxis=dict(tickfont=dict(size=11)),
    )
    fig_sa.show()

In [13]:
# ── 5b: Magnitude TGA heatmap per graph ───────────────────────────────────────
for g in DISC_GRAPHS:
    sub = df_tga[df_tga["Graph"] == g]
    pivot = sub.pivot_table(index="Method", columns="Dataset", values="MeanTGA")
    pivot = pivot[[DS_LABELS[ds] for ds in DATASETS if DS_LABELS[ds] in pivot.columns]]

    abs_max_tga = float(np.nanmax(np.abs(pivot.values)))

    fig_tga = go.Figure(go.Heatmap(
        z            = pivot.values,
        x            = pivot.columns.tolist(),
        y            = pivot.index.tolist(),
        colorscale   = "RdBu",
        zmid=0,
        zmin=-abs_max_tga, zmax=abs_max_tga,
        text         = [[f"{v:.4f}" if not np.isnan(v) else "—" for v in row] for row in pivot.values],
        texttemplate = "%{text}",
        textfont     = dict(size=12),
        colorbar     = dict(title=dict(text="Mean TGA", font=dict(size=11))),
        hovertemplate=f"<b>%{{y}}</b> ({g}) — %{{x}}<br>TGA=%{{z:.4f}}<extra></extra>",
    ))
    fig_tga.update_layout(
        title=dict(text=f"Magnitude TGA vs True DAG — {g} graph<br>"
                   "<sup>Signed: positive = disc overestimates True; negative = underestimates; white ≈ 0 = matches</sup>",
                   font=dict(size=12)),
        height=260, width=700,
        margin=dict(l=130, r=80, t=70, b=60),
        xaxis=dict(tickfont=dict(size=11), tickangle=-15),
        yaxis=dict(tickfont=dict(size=11)),
    )
    fig_tga.show()


In [14]:
# ── 5c: Summary scatter — TGA vs Sign Disagreement (True DAG) ────────────────
# Color = Method, Shape = Graph, all datasets combined
# X-axis: TGA, Y-axis: 1 - Sign Alignment (sign disagreement rate)
GRAPH_MARKERS = {"PC": "circle", "LiNGAM": "diamond"}

fig_summary_true = go.Figure()
SEEN_METHODS_TRUE = set()

df_summary_true = df_sa[["Dataset", "Method", "Graph", "MeanSignAlign"]].merge(
    df_tga[["Dataset", "Method", "Graph", "MeanTGA"]],
    on=["Dataset", "Method", "Graph"]
)

for _, row in df_summary_true.iterrows():
    meth = row["Method"]
    g    = row["Graph"]
    ds   = row["Dataset"]
    show_meth = meth not in SEEN_METHODS_TRUE
    SEEN_METHODS_TRUE.add(meth)
    
    sign_disagree = 1 - row["MeanSignAlign"]
    
    fig_summary_true.add_trace(go.Scatter(
        x    = [row["MeanTGA"]],
        y    = [sign_disagree],
        mode = "markers",
        name = meth,
        legendgroup = meth,
        showlegend  = show_meth,
        marker = dict(
            color  = METHOD_COLORS[meth],
            symbol = GRAPH_MARKERS[g],
            size   = 11,
            line   = dict(width=1.0, color="white"),
        ),
        hovertemplate=(
            f"<b>{meth}</b> — {g}<br>"
            f"Dataset: {ds}<br>"
            "TGA: %{x:.4f}<br>"
            "Sign disagreement (1−align): %{y:.3f}<extra></extra>"
        ),
    ))

# Graph shape legend (invisible traces)
for g_label, symbol in GRAPH_MARKERS.items():
    fig_summary_true.add_trace(go.Scatter(
        x=[None], y=[None], mode="markers",
        name=g_label,
        legendgroup=f"graph_{g_label}",
        showlegend=True,
        marker=dict(color="gray", symbol=symbol, size=11),
    ))

# Reference line at TGA = 0
fig_summary_true.add_vline(x=0, line=dict(dash="dot", color="lightgray", width=1.5))
fig_summary_true.add_annotation(
    x=0, y=0.48, text="TGA = 0 (matches True)", showarrow=False,
    font=dict(size=9, color="#aaaaaa"), xanchor="center", yanchor="bottom",
)

fig_summary_true.update_layout(
    title=dict(
        text="True-Graph Alignment Summary — All Datasets<br>"
             "<sup>Colour = method, Shape = graph | Lower disagreement + TGA near 0 = closer to True oracle</sup>",
        font=dict(size=12),
    ),
    xaxis=dict(
        title="Magnitude TGA vs True DAG",
        zeroline=True, zerolinewidth=1.5, zerolinecolor="gray",
    ),
    yaxis=dict(
        title="1 − Sign Alignment (sign disagreement rate)",
        range=[0, 0.5],
    ),
    height=480, width=660,
    legend=dict(x=1.02, y=1, xanchor="left", font=dict(size=11)),
    margin=dict(l=70, r=180, t=90, b=70),

)
fig_summary_true.show()

In [15]:
# ── Summary Table: Mean Metrics by Method & Graph (True DAG) ─────────────────
df_summary_true_agg = df_summary_true.groupby(["Method", "Graph"], as_index=False).agg({
    "MeanSignAlign": "mean",
    "MeanTGA": "mean"
})
df_summary_true_agg["SignDisagreement"] = 1 - df_summary_true_agg["MeanSignAlign"]
df_summary_true_agg = df_summary_true_agg.rename(columns={
    "MeanSignAlign": "Mean Sign Alignment",
    "MeanTGA": "Mean TGA",
    "SignDisagreement": "Mean Sign Disagreement (1−SA)"
})
df_summary_true_agg = df_summary_true_agg.sort_values(["Graph", "Method"])

print("\nSummary: Mean Metrics vs True DAG (averaged across all datasets)")
print("=" * 80)
display(df_summary_true_agg.round(4))


Summary: Mean Metrics vs True DAG (averaged across all datasets)


,Method,Graph,Mean Sign Alignment,Mean TGA,Mean Sign Disagreement (1−SA)
0,Asymmetric,LiNGAM,0.9370,0.0008,0.0630
2,Causal,LiNGAM,0.6254,-0.0287,0.3746
4,Flow,LiNGAM,0.5509,0.0230,0.4491
1,Asymmetric,PC,0.9412,0.0003,0.0588
3,Causal,PC,0.6840,-0.0218,0.3160
5,Flow,PC,0.5774,0.0343,0.4226


## 6. Methods vs Traditional Baseline

Compares each (method × graph) against **Traditional** (graph-free SHAP) using the same sign-alignment and magnitude TGA metrics as Section 5.  
This measures how much the causal graph structure *changes* the attributions relative to the Traditional baseline, across all three graph variants (PC, LiNGAM, True).

- **Sign Alignment vs Traditional** — fraction of (instance, feature) pairs where `sign(φ_method)` matches `sign(φ_Traditional)`. Higher = output signs closer to graph-free baseline.  
- **Magnitude TGA vs Traditional** — relative magnitude deviation from Traditional. Lower = magnitudes closer to graph-free baseline.


In [16]:
# ── Metrics already computed in section 2 ─────────────────────────────────────
# Using df_sa_scratch and df_tga_scratch for visualizations

print(f"Traditional baseline metrics (computed in section 2):")
print(f"  Sign Alignment vs Scratch: {len(df_sa_scratch)} rows")
print(f"  TGA vs Scratch: {len(df_tga_scratch)} rows")
print("\nSign Alignment vs Traditional (sample):")
print(df_sa_scratch.pivot_table(index="Method", columns=["Graph", "Dataset"], values="MeanSignAlign").round(3).head())

Traditional baseline metrics (computed in section 2):
  Sign Alignment vs Scratch: 6 rows
  TGA vs Scratch: 6 rows

Sign Alignment vs Traditional (sample):
Graph        LiNGAM       PC
Dataset    Lin-Conf Lin-Conf
Method                      
Asymmetric    0.942    0.949
Causal        0.661    0.647
Flow          0.623    0.634


In [17]:
# ── 6a: Sign Alignment vs Traditional — (Method, Graph) × Dataset heatmap ────
pivot_sa_scratch = df_sa_scratch.pivot_table(
    index   = ["Method", "Graph"],
    columns = "Dataset",
    values  = "MeanSignAlign",
)
pivot_sa_scratch = pivot_sa_scratch[
    [DS_LABELS[ds] for ds in DATASETS if DS_LABELS[ds] in pivot_sa_scratch.columns]
]
# Sort rows: group by method, order graphs as PC → LiNGAM → True
graph_order = {"PC": 0, "LiNGAM": 1, "True": 2}
pivot_sa_scratch = pivot_sa_scratch.loc[
    sorted(pivot_sa_scratch.index, key=lambda t: (t[0], graph_order.get(t[1], 9)))
]

row_labels = [f"{m} ({g})" for m, g in pivot_sa_scratch.index]
col_labels = pivot_sa_scratch.columns.tolist()

fig_sa_scratch = go.Figure(go.Heatmap(
    z            = pivot_sa_scratch.values,
    x            = col_labels,
    y            = row_labels,
    colorscale   = "Blues",
    zmin=0.5, zmax=1.0,
    text         = [[f"{v:.3f}" if not np.isnan(v) else "—" for v in row]
                    for row in pivot_sa_scratch.values],
    texttemplate = "%{text}",
    textfont     = dict(size=12),
    colorbar     = dict(title=dict(text="Sign Align<br>vs Traditional", font=dict(size=10))),
    hovertemplate="<b>%{y}</b> — %{x}<br>Sign Align vs Traditional = %{z:.3f}<extra></extra>",
))
fig_sa_scratch.update_layout(
    title=dict(
        text="Sign Alignment vs Traditional Baseline<br>"
             "<sup>Fraction of instances/features where sign agrees with graph-free output</sup>",
        font=dict(size=13),
    ),
    height=400, width=700,
    margin=dict(l=170, r=100, t=80, b=60),
    xaxis=dict(tickfont=dict(size=11), tickangle=-15),
    yaxis=dict(tickfont=dict(size=10)),
)
fig_sa_scratch.show()


In [18]:
# ── 6b: Magnitude TGA vs Traditional — (Method, Graph) × Dataset heatmap ─────
pivot_tga_scratch = df_tga_scratch.pivot_table(
    index   = ["Method", "Graph"],
    columns = "Dataset",
    values  = "MeanTGA",
)
pivot_tga_scratch = pivot_tga_scratch[
    [DS_LABELS[ds] for ds in DATASETS if DS_LABELS[ds] in pivot_tga_scratch.columns]
]
pivot_tga_scratch = pivot_tga_scratch.loc[
    sorted(pivot_tga_scratch.index, key=lambda t: (t[0], graph_order.get(t[1], 9)))
]

row_labels_tga = [f"{m} ({g})" for m, g in pivot_tga_scratch.index]
abs_max_scratch = float(np.nanmax(np.abs(pivot_tga_scratch.values)))

fig_tga_scratch = go.Figure(go.Heatmap(
    z            = pivot_tga_scratch.values,
    x            = pivot_tga_scratch.columns.tolist(),
    y            = row_labels_tga,
    colorscale   = "RdBu",
    zmid=0,
    zmin=-abs_max_scratch, zmax=abs_max_scratch,
    text         = [[f"{v:.4f}" if not np.isnan(v) else "—" for v in row]
                    for row in pivot_tga_scratch.values],
    texttemplate = "%{text}",
    textfont     = dict(size=12),
    colorbar     = dict(title=dict(text="TGA<br>vs Traditional", font=dict(size=10))),
    hovertemplate="<b>%{y}</b> — %{x}<br>TGA vs Traditional = %{z:.4f}<extra></extra>",
))
fig_tga_scratch.update_layout(
    title=dict(
        text="Magnitude TGA vs Traditional Baseline<br>"
             "<sup>Signed: positive = method overestimates Traditional; negative = underestimates; white ≈ 0 = matches</sup>",
        font=dict(size=13),
    ),
    height=400, width=700,
    margin=dict(l=170, r=100, t=80, b=60),
    xaxis=dict(tickfont=dict(size=11), tickangle=-15),
    yaxis=dict(tickfont=dict(size=10)),
)
fig_tga_scratch.show()


In [19]:
# ── 6c: Summary scatter — TGA vs Sign Disagreement (Traditional baseline) ───
# X-axis: TGA, Y-axis: 1 - Sign Alignment (sign disagreement rate)
fig_summary_scratch = go.Figure()
SEEN_METHODS_SCRATCH = set()

df_summary_scratch = df_sa_scratch[["Dataset", "Method", "Graph", "MeanSignAlign"]].merge(
    df_tga_scratch[["Dataset", "Method", "Graph", "MeanTGA"]],
    on=["Dataset", "Method", "Graph"]
)

for _, row in df_summary_scratch.iterrows():
    meth = row["Method"]
    g    = row["Graph"]
    ds   = row["Dataset"]
    show_meth = meth not in SEEN_METHODS_SCRATCH
    SEEN_METHODS_SCRATCH.add(meth)
    
    sign_disagree = 1 - row["MeanSignAlign"]
    
    fig_summary_scratch.add_trace(go.Scatter(
        x    = [row["MeanTGA"]],
        y    = [sign_disagree],
        mode = "markers",
        name = meth,
        legendgroup = meth,
        showlegend  = show_meth,
        marker = dict(
            color  = METHOD_COLORS[meth],
            symbol = GRAPH_MARKERS[g],
            size   = 11,
            line   = dict(width=1.0, color="white"),
        ),
        hovertemplate=(
            f"<b>{meth}</b> — {g}<br>"
            f"Dataset: {ds}<br>"
            "TGA: %{x:.4f}<br>"
            "Sign disagreement (1−align): %{y:.3f}<extra></extra>"
        ),
    ))

# Graph shape legend
for g_label, symbol in GRAPH_MARKERS.items():
    fig_summary_scratch.add_trace(go.Scatter(
        x=[None], y=[None], mode="markers",
        name=g_label,
        legendgroup=f"graph_{g_label}",
        showlegend=True,
        marker=dict(color="gray", symbol=symbol, size=11),
    ))

# Reference line at TGA = 0
fig_summary_scratch.add_vline(x=0, line=dict(dash="dot", color="lightgray", width=1.5))
fig_summary_scratch.add_annotation(
    x=0, y=0.48, text="TGA = 0 (matches Traditional)", showarrow=False,
    font=dict(size=9, color="#aaaaaa"), xanchor="center", yanchor="bottom",
)

fig_summary_scratch.update_layout(
    title=dict(
        text="Traditional Baseline Alignment Summary — All Datasets<br>"
             "<sup>Colour = method, Shape = graph | Lower disagreement + TGA near 0 = closer to Traditional baseline</sup>",
        font=dict(size=12),
    ),
    xaxis=dict(
        title="Magnitude TGA vs Traditional",
        zeroline=True, zerolinewidth=1.5, zerolinecolor="gray",
    ),
    yaxis=dict(
        title="1 − Sign Alignment (sign disagreement rate)",
        range=[0, 0.5],
    ),
    height=480, width=660,
    legend=dict(x=1.02, y=1, xanchor="left", font=dict(size=11)),
    margin=dict(l=70, r=180, t=90, b=70),

)
fig_summary_scratch.show()

In [20]:
# ── Summary Table: Mean Metrics by Method & Graph (Traditional baseline) ─────
df_summary_scratch_agg = df_summary_scratch.groupby(["Method", "Graph"], as_index=False).agg({
    "MeanSignAlign": "mean",
    "MeanTGA": "mean"
})
df_summary_scratch_agg["SignDisagreement"] = 1 - df_summary_scratch_agg["MeanSignAlign"]
df_summary_scratch_agg = df_summary_scratch_agg.rename(columns={
    "MeanSignAlign": "Mean Sign Alignment",
    "MeanTGA": "Mean TGA",
    "SignDisagreement": "Mean Sign Disagreement (1−SA)"
})
df_summary_scratch_agg = df_summary_scratch_agg.sort_values(["Graph", "Method"])

print("\nSummary: Mean Metrics vs Traditional Baseline (averaged across all datasets)")
print("=" * 80)
display(df_summary_scratch_agg.round(4))


Summary: Mean Metrics vs Traditional Baseline (averaged across all datasets)


,Method,Graph,Mean Sign Alignment,Mean TGA,Mean Sign Disagreement (1−SA)
0,Asymmetric,LiNGAM,0.9424,0.0010,0.0576
2,Causal,LiNGAM,0.6608,0.0742,0.3392
4,Flow,LiNGAM,0.6232,0.0291,0.3768
1,Asymmetric,PC,0.9492,0.0005,0.0508
3,Causal,PC,0.6470,0.0812,0.3530
5,Flow,PC,0.6342,0.0403,0.3658


## 7. Comprehensive Summary Table

All metrics in one pivoted table — easy to copy into the thesis.

In [21]:
rows_summary = []

for ds in DATASETS:
    d             = all_data[ds]
    shap_data     = d["shap_data"]
    feature_names = d["feature_names"]
    y_parent_set  = d["y_parent_set"]
    n_feat        = len(feature_names)
    random_base   = len(y_parent_set) / n_feat
    scratch_ma    = mean_abs_shap(shap_data, "Scratch")

    for meth in BASE_METHODS:
        for g in DISC_GRAPHS:
            key = f"{meth} ({g})"
            if key not in shap_data:
                continue
            ma = mean_abs_shap(shap_data, key)
            rho, _ = spearmanr(ma, scratch_ma)

            j_k10 = top_k_jaccard(ma, scratch_ma, 10)
            prec_k10_set = set(np.argsort(ma)[-10:])
            prec_k10 = len(prec_k10_set & y_parent_set) / 10

            # GSS — signed magnitude difference (no outer abs, no normalisation)
            pc_k = f"{meth} (PC)"
            lg_k = f"{meth} (LiNGAM)"
            gss_val = np.nan
            if pc_k in shap_data and lg_k in shap_data:
                raw = (np.abs(shap_data[pc_k]) - np.abs(shap_data[lg_k])).mean(axis=0)
                gss_val = float(np.mean(raw))

            # Sign Alignment
            sa_row = df_sa[(df_sa["Dataset"] == DS_LABELS[ds]) &
                           (df_sa["Method"] == meth) & (df_sa["Graph"] == g)]
            sa_val = float(sa_row["MeanSignAlign"].values[0]) if not sa_row.empty else np.nan

            # TGA
            tga_row = df_tga[(df_tga["Dataset"] == DS_LABELS[ds]) &
                             (df_tga["Method"] == meth) & (df_tga["Graph"] == g)]
            tga_val = float(tga_row["MeanTGA"].values[0]) if not tga_row.empty else np.nan

            rows_summary.append(dict(
                Dataset    = DS_LABELS[ds],
                Method     = meth,
                Graph      = g,
                Spearman_ρ = round(float(rho), 3),
                Jaccard_10 = round(float(j_k10), 3),
                Prec_10    = round(float(prec_k10), 3),
                Rand_Base  = round(float(random_base), 3),
                GSS        = round(float(gss_val), 4) if not np.isnan(gss_val) else np.nan,
                SignAlign  = round(float(sa_val), 3)  if not np.isnan(sa_val) else np.nan,
                MagTGA     = round(float(tga_val), 4) if not np.isnan(tga_val) else np.nan,
            ))

df_summary = (
    pd.DataFrame(rows_summary)
    .sort_values(["Dataset", "Graph", "Method"])
    .reset_index(drop=True)
)

pd.set_option("display.max_rows", 100)
pd.set_option("display.float_format", "{:.4f}".format)
pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 120)

print(f"Total rows: {len(df_summary)}")
df_summary


Total rows: 6


,Dataset,Method,Graph,Spearman_ρ,Jaccard_10,Prec_10,Rand_Base,GSS,SignAlign,MagTGA
0,Lin-Conf,Asymmetric,LiNGAM,0.9950,1.0000,0.9000,0.3000,-0.0005,0.9370,0.0008
1,Lin-Conf,Causal,LiNGAM,0.5910,0.5380,0.7000,0.3000,0.0070,0.6250,-0.0287
2,Lin-Conf,Flow,LiNGAM,0.6330,0.4290,0.7000,0.3000,0.0112,0.5510,0.0230
3,Lin-Conf,Asymmetric,PC,0.9980,1.0000,0.9000,0.3000,-0.0005,0.9410,0.0003
4,Lin-Conf,Causal,PC,0.5450,0.2500,0.6000,0.3000,0.0070,0.6840,-0.0218
5,Lin-Conf,Flow,PC,0.5710,0.3330,0.5000,0.3000,0.0112,0.5770,0.0343


## Section 8 — Per-Feature Diagnostic: Top-K Most Disagreeing / Most Stable Features

Surface the features at **both extremes** of each metric for targeted qualitative analysis.

### 8a–8e — Most unstable / most disagreeing features

| Sub-section | Question answered |
|-------------|-------------------|
| **8a** | Which features shift the most in *importance rank* when switching from Traditional to Method/Graph? |
| **8b** | Which features have the highest *GSS* — largest magnitude instability between PC and LiNGAM? |
| **8c** | Which features have the lowest *SSS* — most sign-unstable between PC and LiNGAM? |
| **8d** | Which features have the lowest *Sign Alignment* vs the True DAG? |
| **8e** | Which features have the highest *Magnitude TGA* vs the True DAG? |

### 8f–8i — Most stable / least-changed features

| Sub-section | Question answered |
|-------------|-------------------|
| **8f** | Which features have the *smallest rank shift* — importance barely changes when adding causal graph info? |
| **8g** | Which features have the lowest *GSS* — magnitude stays most consistent between PC and LiNGAM? |
| **8h** | Which features have the highest *SSS* — sign stays most consistent between PC and LiNGAM? |
| **8i** | Which features have the lowest *Magnitude TGA* — magnitudes closest to the True DAG oracle? |

`K_DIAG = 5` throughout (change the variable in cell 8a to adjust all sub-sections).


In [22]:
# Optional: export the summary table to CSV
out_path = BASE_DIR / "data" / "explainability" / "shapley_summary_metrics.csv"
df_summary.to_csv(out_path, index=False)
print(f"Saved → {out_path}")

Saved → /Users/juanrios/Documents/master_thesis/data/explainability/shapley_summary_metrics.csv


In [23]:
details=feature_profile("X37", "Asymmetric (LiNGAM)")


════════════════════════════════════════════════════════════════════════
  FEATURE PROFILE: X37  |  Asymmetric (LiNGAM)  |  Lin-Conf
════════════════════════════════════════════════════════════════════════

  CORE ATTRIBUTION
    Mean SHAP (signed)  :  +0.008866
    Mean |SHAP|         :   0.058858  →  Rank 23/50
    Is Y-parent (true)  :   No
    Rank vs Scratch     :   23  →  23  (0  = no change)

  GRAPH SENSITIVITY  (Asymmetric: PC vs LiNGAM)
    GSS (magnitude)    :  -0.000707  (LiNGAM assigns more weight)
    SSS (sign)         :   0.9500  (stable)

  ALIGNMENT VS TRUE DAG  (Asymmetric — LiNGAM)
    Sign Alignment     :   0.9500  (agrees with true oracle)
    TGA                :  +0.002213  (over-estimates true-graph magnitude)

  ALIGNMENT VS SCRATCH  (Asymmetric — LiNGAM)
    Sign Alignment     :   0.9600  (agrees with scratch baseline)
    TGA                :  +0.002390  (over-estimates scratch magnitude)

  RANK COMPARISON  (all methods, feature: X37)


,Method,Rank,Mean SHAP,Mean |SHAP|,Current →
1,Causal (True),35,0.0061,0.1132,
2,Causal (PC),32,-0.0146,0.1045,
3,Asymmetric (LiNGAM),23,0.0089,0.0589,←
4,Asymmetric (PC),23,0.0123,0.0582,
5,Asymmetric (True),23,0.0083,0.0566,
6,Scratch,23,0.0123,0.0565,
7,Causal (LiNGAM),44,0.0127,0.0534,
8,Flow (PC),32,0.0091,0.0477,
9,Flow (True),38,0.0033,0.0446,
10,Flow (LiNGAM),44,0.0044,0.0167,


In [24]:
K_DIAG = 5

In [25]:
# ── 8b  Top-K features by GSS (magnitude instability PC vs LiNGAM) ───────────
# Ranked by |GSS| (largest magnitude change regardless of sign).
# Signed value in the table shows direction: positive = PC > LiNGAM, negative = LiNGAM > PC.

for ds in DATASETS:
    label      = DS_LABELS[ds]
    shap_d     = all_data[ds]["shap_data"]
    feat_names = all_data[ds]["feature_names"]

    print(f"\n{'═' * 72}")
    print(f"  {label}  ·  Top-{K_DIAG} features by |GSS|  (highest magnitude instability PC vs LiNGAM)")
    print(f"  Positive GSS = PC assigns higher importance;  Negative GSS = LiNGAM higher")
    print(f"{'═' * 72}")
    for meth, arr in gss_feat.items():
        df_g = top_k_features(arr, feat_names, k=K_DIAG, ascending=False, score_col="GSS")
        print(f"\n  {meth}")
        display(df_g)



════════════════════════════════════════════════════════════════════════
  Lin-Conf  ·  Top-5 features by |GSS|  (highest magnitude instability PC vs LiNGAM)
  Positive GSS = PC assigns higher importance;  Negative GSS = LiNGAM higher
════════════════════════════════════════════════════════════════════════

  Asymmetric


,feature,GSS
rank,,
1,X33,-0.0858
2,X41,0.0535
3,X47,0.0353
4,X7,-0.0303
5,X24,0.0174



  Causal


,feature,GSS
rank,,
1,X2,-0.9103
2,X47,-0.9062
3,X21,0.7625
4,X33,0.7242
5,X30,0.7156



  Flow


,feature,GSS
rank,,
1,X33,1.7085
2,X47,-1.2893
3,X46,-0.6322
4,X45,0.6141
5,X32,0.4655


In [26]:
# ── 8c  Bottom-K features by SSS (sign instability between PC and LiNGAM) ────
# Lower SSS = that feature flips sign more often when you swap PC for LiNGAM.

for ds in DATASETS:
    label      = DS_LABELS[ds]
    shap_d     = all_data[ds]["shap_data"]
    feat_names = all_data[ds]["feature_names"]

    print(f"\n{'═' * 72}")
    print(f"  {label}  ·  Bottom-{K_DIAG} features by SSS  (lower = more sign-unstable)")
    print(f"{'═' * 72}")
    for meth, arr in sss_feat.items():
        df_s = top_k_features(arr, feat_names, k=K_DIAG, ascending=True, score_col="SSS")
        print(f"\n  {meth}")
        display(df_s)



════════════════════════════════════════════════════════════════════════
  Lin-Conf  ·  Bottom-5 features by SSS  (lower = more sign-unstable)
════════════════════════════════════════════════════════════════════════

  Asymmetric


,feature,SSS
rank,,
1,X13,0.7200
2,X29,0.8100
3,X3,0.8163
4,X9,0.8200
5,X34,0.8300



  Causal


,feature,SSS
rank,,
1,X8,0.3600
2,X32,0.3700
3,X10,0.4200
4,X30,0.4300
5,X11,0.4500



  Flow


,feature,SSS
rank,,
1,X10,0.4343
2,X5,0.5300
3,X26,0.5341
4,X13,0.5875
5,X43,0.6615


In [27]:
# ── 8d  Bottom-K features by Sign Alignment vs True DAG ──────────────────────
# Lowest sign alignment = the method's directional attribution most often disagrees
# with the oracle (True DAG) for that feature.

for ds in DATASETS:
    label       = DS_LABELS[ds]
    sub_shap_d  = all_data[ds]["subset_shap_data"]
    feat_names  = all_data[ds]["feature_names"]
    # sa_feat     = compute_sign_alignment(sub_shap_d, BASE_METHODS, DISC_GRAPHS, reference="True")

    print(f"\n{'═' * 72}")
    print(f"  {label}  ·  Bottom-{K_DIAG} features by Sign Alignment vs True DAG")
    print(f"  (lower = method/graph attributions most often flip sign vs True-DAG oracle)")
    print(f"{'═' * 72}")
    for (meth, g), arr in sign_align.items():
        df_sa_f = top_k_features(arr, feat_names, k=K_DIAG, ascending=True, score_col="SignAlign")
        print(f"\n  {meth} ({g})")
        display(df_sa_f)



════════════════════════════════════════════════════════════════════════
  Lin-Conf  ·  Bottom-5 features by Sign Alignment vs True DAG
  (lower = method/graph attributions most often flip sign vs True-DAG oracle)
════════════════════════════════════════════════════════════════════════

  Asymmetric (PC)


,feature,SignAlign
rank,,
1,X27,0.5700
2,X13,0.7200
3,X29,0.8100
4,X3,0.8100
5,X9,0.8100



  Asymmetric (LiNGAM)


,feature,SignAlign
rank,,
1,X27,0.5900
2,X9,0.6300
3,X3,0.7300
4,X34,0.7700
5,X10,0.8500



  Causal (PC)


...

,feature,SignAlign
rank,,
1,X42,0.3500
2,X32,0.3800
3,X36,0.4000
4,X10,0.4100
5,X30,0.4200



  Flow (PC)


,feature,SignAlign
rank,,
1,X27,0.2184
2,X36,0.2727
3,X35,0.3700
4,X34,0.3737
5,X8,0.3776



  Flow (LiNGAM)


,feature,SignAlign
rank,,
1,X27,0.2069
2,X36,0.2828
3,X14,0.3030
4,X16,0.3200
5,X8,0.3571


In [28]:
# ── 8e  Top-K features by Magnitude TGA vs True DAG ──────────────────────────
# Highest TGA = the method's attribution magnitudes deviate the most from the
# oracle (True DAG) for that feature; TGA > 1 means the error exceeds the reference mean.
K_DIAG = 5
for ds in DATASETS:
    label       = DS_LABELS[ds]
    sub_shap_d  = all_data[ds]["subset_shap_data"]
    feat_names  = all_data[ds]["feature_names"]

    print(f"\n{'═' * 72}")
    print(f"  {label}  ·  Top-{K_DIAG} features by Magnitude TGA vs True DAG")
    print(f"  (higher = absolute magnitudes furthest from the True-DAG oracle;")
    print(f"   TGA > 1 means error exceeds the oracle's own mean attribution)")
    print(f"{'═' * 72}")
    for (meth, g), arr in tga_data.items():
        df_tga_f = top_k_features(arr, feat_names, k=K_DIAG, ascending=False, score_col="TGA")
        print(f"\n  {meth} ({g})")
        display(df_tga_f)



════════════════════════════════════════════════════════════════════════
  Lin-Conf  ·  Top-5 features by Magnitude TGA vs True DAG
  (higher = absolute magnitudes furthest from the True-DAG oracle;
   TGA > 1 means error exceeds the oracle's own mean attribution)
════════════════════════════════════════════════════════════════════════

  Asymmetric (PC)


,feature,TGA
rank,,
1,X33,-0.0851
2,X41,0.0379
3,X7,-0.0170
4,X6,0.0154
5,X47,0.0150



  Asymmetric (LiNGAM)


,feature,TGA
rank,,
1,X47,-0.0203
2,X41,-0.0156
3,X7,0.0133
4,X6,0.0114
5,X17,0.0111



  Causal (PC)


...

,feature,TGA
rank,,
1,X6,-0.9963
2,X24,0.9672
3,X47,0.9003
4,X21,-0.7486
5,X45,-0.7428



  Flow (PC)


,feature,TGA
rank,,
1,X33,1.6441
2,X24,1.3265
3,X4,0.3960
4,X42,-0.3629
5,X31,-0.3418



  Flow (LiNGAM)


,feature,TGA
rank,,
1,X24,1.3822
2,X47,1.3096
3,X46,0.6468
4,X45,-0.6153
5,X48,0.5101


In [29]:
# ── 8f  Top-K features by Magnitude TGA vs Traditional DAG ──────────────────────────
# Highest TGA = the method's attribution magnitudes deviate the most from the
# oracle (Traditional DAG) for that feature; TGA > 1 means the error exceeds the reference mean.
K_DIAG = 5
for ds in DATASETS:
    label       = DS_LABELS[ds]
    sub_shap_d  = all_data[ds]["subset_shap_data"]
    feat_names  = all_data[ds]["feature_names"]

    print(f"\n{'═' * 72}")
    print(f"  {label}  ·  Top-{K_DIAG} features by Magnitude TGA vs Traditional DAG")
    print(f"  (higher = absolute magnitudes furthest from the Traditional shapley value;")
    print(f"   TGA > 1 means error exceeds the oracle's own mean attribution)")
    print(f"{'═' * 72}")
    for (meth, g), arr in tga_scratch.items():
        df_tga_f = top_k_features(arr, feat_names, k=K_DIAG, ascending=False, score_col="TGA")
        print(f"\n  {meth} ({g})")
        display(df_tga_f)



════════════════════════════════════════════════════════════════════════
  Lin-Conf  ·  Top-5 features by Magnitude TGA vs Traditional DAG
  (higher = absolute magnitudes furthest from the Traditional shapley value;
   TGA > 1 means error exceeds the oracle's own mean attribution)
════════════════════════════════════════════════════════════════════════

  Asymmetric (PC)


,feature,TGA
rank,,
1,X33,-0.0417
2,X41,0.0381
3,X47,0.0280
4,X7,-0.0136
5,X24,-0.0120



  Asymmetric (LiNGAM)


,feature,TGA
rank,,
1,X33,0.0441
2,X24,-0.0294
3,X7,0.0167
4,X41,-0.0154
5,X40,0.0146



  Causal (PC)


...

,feature,TGA
rank,,
1,X24,1.2795
2,X2,1.0588
3,X33,-0.5685
4,X31,0.4270
5,X7,-0.4210



  Flow (PC)


,feature,TGA
rank,,
1,X47,-0.7010
2,X45,0.5978
3,X21,0.5736
4,X33,0.5237
5,X32,0.5015



  Flow (LiNGAM)


,feature,TGA
rank,,
1,X33,-1.1849
2,X47,0.5884
3,X7,-0.5199
4,X24,0.4875
5,X39,-0.3931


In [57]:
feature_profile("X46", "Flow (LiNGAM)")


════════════════════════════════════════════════════════════════════════
  FEATURE PROFILE: X46  |  Flow (LiNGAM)  |  Lin-Conf
════════════════════════════════════════════════════════════════════════

  CORE ATTRIBUTION
    Mean SHAP (signed)  :  +0.005293
    Mean |SHAP|         :   1.194207  →  Rank 3/50
    Is Y-parent (true)  :   Yes ✓
    Rank vs Scratch     :   4  →  3  (-1  ↑ more important)

  GRAPH SENSITIVITY  (Flow: PC vs LiNGAM)
    GSS (magnitude)    :  -0.632160  (LiNGAM assigns more weight)
    SSS (sign)         :   0.9892  (stable)

  ALIGNMENT VS TRUE DAG  (Flow — LiNGAM)
    Sign Alignment     :   0.6429  (partially agrees with true oracle)
    TGA                :  +0.646764  (over-estimates true-graph magnitude)

  ALIGNMENT VS SCRATCH  (Flow — LiNGAM)
    Sign Alignment     :   0.7700  (partially agrees with scratch baseline)
    TGA                :  +0.328555  (over-estimates scratch magnitude)

  RANK COMPARISON  (all methods, feature: X46)


,Method,Rank,Mean SHAP,Mean |SHAP|,Current →
1,Flow (LiNGAM),3,0.0053,1.1942,←
2,Causal (LiNGAM),4,-0.2263,0.9369,
3,Scratch,4,-0.2144,0.8657,
4,Asymmetric (PC),4,-0.2034,0.8626,
5,Asymmetric (LiNGAM),4,-0.2079,0.8591,
6,Asymmetric (True),4,-0.2065,0.8524,
7,Flow (PC),7,0.0110,0.5620,
8,Flow (True),5,-0.0876,0.5474,
9,Causal (True),14,-0.1470,0.4447,
10,Causal (PC),13,-0.1539,0.4097,


,Method,Rank,Mean SHAP,Mean |SHAP|,Current →
1,Flow (LiNGAM),3,0.0053,1.1942,←
2,Causal (LiNGAM),4,-0.2263,0.9369,
3,Scratch,4,-0.2144,0.8657,
4,Asymmetric (PC),4,-0.2034,0.8626,
5,Asymmetric (LiNGAM),4,-0.2079,0.8591,
6,Asymmetric (True),4,-0.2065,0.8524,
7,Flow (PC),7,0.0110,0.5620,
8,Flow (True),5,-0.0876,0.5474,
9,Causal (True),14,-0.1470,0.4447,
10,Causal (PC),13,-0.1539,0.4097,


---
### 8f–8i — Most Stable / Least-Changed Features


In [30]:
# ── 8g  Lowest |GSS| features (most stable magnitude between PC and LiNGAM) ──
# Ranked by |GSS| ascending → features closest to zero = PC and LiNGAM agree most.
# Signed value shows any residual direction (near-zero positive or negative).

for ds in DATASETS:
    label      = DS_LABELS[ds]
    shap_d     = all_data[ds]["shap_data"]
    feat_names = all_data[ds]["feature_names"]
    gss_feat   = compute_gss(shap_d, BASE_METHODS)

    print(f"\n{'═' * 72}")
    print(f"  {label}  ·  Bottom-{K_DIAG} features by |GSS|  (lowest magnitude = most stable PC vs LiNGAM)")
    print(f"{'═' * 72}")
    for meth, arr in gss_feat.items():
        df_g = top_k_features(arr, feat_names, k=K_DIAG, ascending=True, score_col="GSS")
        print(f"\n  {meth}")
        display(df_g)



════════════════════════════════════════════════════════════════════════
  Lin-Conf  ·  Bottom-5 features by |GSS|  (lowest magnitude = most stable PC vs LiNGAM)
════════════════════════════════════════════════════════════════════════

  Asymmetric


,feature,GSS
rank,,
1,X31,0.0000
2,X35,-0.0000
3,X8,0.0000
4,X28,-0.0002
5,X22,-0.0002



  Causal


,feature,GSS
rank,,
1,X44,0.0003
2,X7,-0.0024
3,X22,0.0028
4,X3,0.0032
5,X11,-0.0037



  Flow


,feature,GSS
rank,,
1,X22,-0.0001
2,X18,-0.0002
3,X11,-0.0005
4,X35,-0.0005
5,X27,0.0006


In [31]:
# ── 8h  Highest SSS features (most sign-stable between PC and LiNGAM) ─────────
# Features whose SHAP sign is most consistently the same under PC and LiNGAM.

for ds in DATASETS:
    label      = DS_LABELS[ds]
    shap_d     = all_data[ds]["shap_data"]
    feat_names = all_data[ds]["feature_names"]
    sss_feat   = compute_sss(shap_d, BASE_METHODS)

    print(f"\n{'═' * 72}")
    print(f"  {label}  ·  Top-{K_DIAG} features by SSS  (higher = most sign-stable PC vs LiNGAM)")
    print(f"{'═' * 72}")
    for meth, arr in sss_feat.items():
        df_s = top_k_features(arr, feat_names, k=K_DIAG, ascending=False, score_col="SSS")
        print(f"\n  {meth}")
        display(df_s)



════════════════════════════════════════════════════════════════════════
  Lin-Conf  ·  Top-5 features by SSS  (higher = most sign-stable PC vs LiNGAM)
════════════════════════════════════════════════════════════════════════

  Asymmetric


,feature,SSS
rank,,
1,X24,1.0000
2,X48,1.0000
3,X22,1.0000
4,X0,1.0000
5,X5,0.9900



  Causal


,feature,SSS
rank,,
1,X24,0.9800
2,X4,0.9000
3,X31,0.8800
4,X2,0.8700
5,X44,0.8500



  Flow


,feature,SSS
rank,,
1,X49,1.0000
2,X39,1.0000
3,X7,1.0000
4,X18,1.0000
5,X48,1.0000


In [32]:
# ── 8i  Lowest TGA features (magnitudes closest to the True DAG oracle) ────────
# Features where the method's attributions are closest in magnitude to the oracle.

for ds in DATASETS:
    label       = DS_LABELS[ds]
    sub_shap_d  = all_data[ds]["subset_shap_data"]
    feat_names  = all_data[ds]["feature_names"]
    tga_feat, _ = compute_tga(sub_shap_d, feat_names, BASE_METHODS, DISC_GRAPHS, reference="True")

    print(f"\n{'═' * 72}")
    print(f"  {label}  ·  Bottom-{K_DIAG} features by Magnitude TGA vs True DAG")
    print(f"  (lower = magnitude closest to the True-DAG oracle)")
    print(f"{'═' * 72}")
    for (meth, g), arr in tga_feat.items():
        df_tga_f = top_k_features(arr, feat_names, k=K_DIAG, ascending=True, score_col="TGA")
        print(f"\n  {meth} ({g})")
        display(df_tga_f)



════════════════════════════════════════════════════════════════════════
  Lin-Conf  ·  Bottom-5 features by Magnitude TGA vs True DAG
  (lower = magnitude closest to the True-DAG oracle)
════════════════════════════════════════════════════════════════════════

  Asymmetric (PC)


,feature,TGA
rank,,
1,X43,0.0000
2,X10,-0.0000
3,X16,-0.0000
4,X35,-0.0001
5,X20,-0.0001



  Asymmetric (LiNGAM)


,feature,TGA
rank,,
1,X25,0.0000
2,X11,-0.0000
3,X35,-0.0001
4,X26,0.0004
5,X31,0.0005



  Causal (PC)


...

,feature,TGA
rank,,
1,X20,-0.0007
2,X11,-0.0052
3,X39,0.0055
4,X25,-0.0056
5,X42,0.0059



  Flow (PC)


,feature,TGA
rank,,
1,X28,-0.0007
2,X11,0.0010
3,X45,-0.0012
4,X23,0.0026
5,X37,0.0031



  Flow (LiNGAM)


,feature,TGA
rank,,
1,X20,0.0013
2,X11,0.0015
3,X9,0.0035
4,X18,0.0049
5,X25,0.0051
